In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv("../data/clean/crop_yield_clean.csv")

conn = sqlite3.connect("../data/crop.db")
df.to_sql("yields", conn, if_exists="replace", index=False)

pd.read_sql("SELECT * FROM yields LIMIT 5", conn)

,year,province,crop,yield_kg_ha
0,2000,Alberta,Barley,3000.0
1,2001,Alberta,Barley,2900.0
2,2002,Alberta,Barley,2300.0
3,2003,Alberta,Barley,3000.0
4,2004,Alberta,Barley,3500.0


In [2]:
pd.read_sql("SELECT COUNT(*) AS total_rows FROM yields", conn)

,total_rows
0,648


In [3]:
with open("../sql/queries.sql") as f:
    queries = [q.strip() for q in f.read().split(";") if q.strip()]

print(len(queries), "queries loaded")

5 queries loaded


In [4]:
pd.read_sql(queries[2], conn)

,crop,province,avg_yield,rnk
0,Barley,Manitoba,3562.3,1
1,Barley,Alberta,3479.0,2
2,Barley,Ontario,3439.5,3
3,Canola,Ontario,2322.0,1
4,Canola,Quebec,2194.2,2
5,Canola,Alberta,2038.3,3
6,Wheat,Ontario,5295.6,1
7,Wheat,Nova Scotia,4195.0,2
8,Wheat,Prince Edward Island,3512.7,3


In [5]:
pd.read_sql(queries[3], conn)

,crop,year,avg_yield,type
0,Barley,2025,3940.0,Best year
1,Barley,2006,2755.6,Worst year
2,Canola,2025,2523.0,Best year
3,Canola,2002,1500.0,Worst year
4,Wheat,2025,4290.8,Best year
5,Wheat,2001,2833.3,Worst year


In [6]:
pd.read_sql("""
SELECT crop, province, COUNT(*) AS years_reported,
       MIN(year) AS first_year, MAX(year) AS last_year,
       ROUND(AVG(yield_kg_ha), 1) AS avg_yield
FROM yields
GROUP BY crop, province
ORDER BY crop, avg_yield DESC
""", conn)

,crop,province,years_reported,first_year,last_year,avg_yield
0,Barley,Manitoba,27,2000,2026,3562.3
1,Barley,Alberta,27,2000,2026,3479.0
2,Barley,Ontario,27,2000,2026,3439.5
3,Barley,Prince Edward Island,27,2000,2026,3248.6
4,Barley,New Brunswick,27,2000,2026,3186.0
5,Barley,Quebec,27,2000,2026,3094.6
6,Barley,Nova Scotia,27,2000,2026,3084.9
7,Barley,Saskatchewan,27,2000,2026,3058.2
8,Barley,British Columbia,27,2000,2026,2985.2
9,Canola,Ontario,27,2000,2026,2322.0


In [7]:
pd.read_sql(queries[0], conn)

,crop,province,avg_yield
0,Barley,Manitoba,3562.3
1,Barley,Alberta,3479.0
2,Barley,Ontario,3439.5
3,Barley,Prince Edward Island,3248.6
4,Barley,New Brunswick,3186.0
5,Barley,Quebec,3094.6
6,Barley,Nova Scotia,3084.9
7,Barley,Saskatchewan,3058.2
8,Barley,British Columbia,2985.2
9,Canola,Ontario,2322.0


In [8]:
pd.read_sql(queries[1], conn).head(15)

,crop,province,year,yield_kg_ha,change_from_prev_year
0,Barley,Alberta,2000,3000.0,NaN
1,Barley,Alberta,2001,2900.0,-100.0
2,Barley,Alberta,2002,2300.0,-600.0
3,Barley,Alberta,2003,3000.0,700.0
4,Barley,Alberta,2004,3500.0,500.0
5,Barley,Alberta,2005,3600.0,100.0
6,Barley,Alberta,2006,3200.0,-400.0
7,Barley,Alberta,2007,3000.0,-200.0
8,Barley,Alberta,2008,3600.0,600.0
9,Barley,Alberta,2009,3100.0,-500.0


In [9]:
pd.read_sql(queries[4], conn).head(15)

,crop,province,year,yield_kg_ha,moving_avg_5yr
0,Barley,Alberta,2000,3000.0,3000.0
1,Barley,Alberta,2001,2900.0,2950.0
2,Barley,Alberta,2002,2300.0,2733.3
3,Barley,Alberta,2003,3000.0,2800.0
4,Barley,Alberta,2004,3500.0,2940.0
5,Barley,Alberta,2005,3600.0,3060.0
6,Barley,Alberta,2006,3200.0,3120.0
7,Barley,Alberta,2007,3000.0,3260.0
8,Barley,Alberta,2008,3600.0,3380.0
9,Barley,Alberta,2009,3100.0,3300.0


In [10]:
conn.close()